**This is the analysis part used for the data gathered during the participant experiment**

In [43]:
#imports
import numpy
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import csv
from pathlib import Path
from IPython.display import display
import analysisfunctions as af

In [44]:
#import data from CSV files
df_eye = pd.read_csv("aggregated_reading_eye_metrics.csv")
df_q = pd.read_csv("aggregated_questionnaire_metrics.csv")

*Check that data imports worked*

In [45]:
issues = []

# ── 1. SHAPE ────────────────────────────────────────────────
print("=== SHAPE ===")
print(f"Questionnaire : {df_q.shape[0]} rows × {df_q.shape[1]} cols")
print(f"Eye metrics   : {df_eye.shape[0]} rows × {df_eye.shape[1]} cols")

# Questionnaire: 1 row per participant → expect 12 rows
# Eye metrics:   3 rows per participant (one per condition) → expect N×3
if df_q.shape[0] != df_q["Participant ID"].nunique():
    issues.append("Questionnaire has duplicate participant rows")

if df_eye.shape[0] % 3 != 0:
    issues.append(f"Eye metrics row count ({df_eye.shape[0]}) is not divisible by 3 — missing condition rows?")


# ── 2. PARTICIPANT ID ALIGNMENT ──────────────────────────────
print("\n=== PARTICIPANT ID ALIGNMENT ===")
q_pids   = set(df_q["Participant ID"].unique())
eye_pids = set(df_eye["Participant ID"].unique())

only_in_q   = q_pids - eye_pids
only_in_eye = eye_pids - q_pids

if only_in_q:
    issues.append(f"Participants in questionnaire but NOT in eye metrics: {sorted(only_in_q)}")
if only_in_eye:
    issues.append(f"Participants in eye metrics but NOT in questionnaire: {sorted(only_in_eye)}")

shared_pids = q_pids & eye_pids
print(f"Shared participants : {len(shared_pids)}")
print(f"Only in questionnaire : {sorted(only_in_q) or 'none'}")
print(f"Only in eye metrics   : {sorted(only_in_eye) or 'none'}")


# ── 3. CONDITION COMPLETENESS (eye metrics) ──────────────────
print("\n=== CONDITION COMPLETENESS ===")
EXPECTED_CONDITIONS = {"none", "full", "eyetracked"}

for pid, grp in df_eye.groupby("Participant ID"):
    found = set(grp["Condition"].dropna().unique())
    missing = EXPECTED_CONDITIONS - found
    if missing:
        issues.append(f"{pid} missing conditions in eye metrics: {missing}")

print("Condition counts per participant:")
print(df_eye.groupby("Participant ID")["Condition"].value_counts().unstack(fill_value=0))


# ── 4. CONDITION LABEL CONSISTENCY ──────────────────────────
print("\n=== CONDITION LABELS ===")
# Questionnaire uses: "none" / "full" / "eyetracked" (in column names as "Bees"/"Hubble"/"Salt")
# Eye metrics Condition column should only contain the three expected values
unexpected_conditions = set(df_eye["Condition"].dropna().unique()) - EXPECTED_CONDITIONS
if unexpected_conditions:
    issues.append(f"Unexpected condition labels in eye metrics: {unexpected_conditions}")
else:
    print("Condition labels OK:", sorted(df_eye["Condition"].dropna().unique()))


# ── 5. MISSING VALUES ────────────────────────────────────────
print("\n=== MISSING VALUES ===")
q_nulls   = df_q.isnull().sum()
eye_nulls = df_eye.isnull().sum()

q_cols_with_nulls   = q_nulls[q_nulls > 0]
eye_cols_with_nulls = eye_nulls[eye_nulls > 0]

if not q_cols_with_nulls.empty:
    issues.append(f"Questionnaire NaNs detected:\n{q_cols_with_nulls}")
    print("Questionnaire NaNs:\n", q_cols_with_nulls)
else:
    print("Questionnaire: no missing values")

if not eye_cols_with_nulls.empty:
    issues.append(f"Eye metrics NaNs detected:\n{eye_cols_with_nulls}")
    print("Eye metrics NaNs:\n", eye_cols_with_nulls)
else:
    print("Eye metrics: no missing values")


# ── 6. DTYPE SANITY ─────────────────────────────────────────
print("\n=== DTYPE CHECKS ===")
# Participant number should be int, age float, scores numeric
EXPECTED_NUMERIC_Q = ["Participant number", "Demographic age"]
for col in EXPECTED_NUMERIC_Q:
    if col in df_q.columns and not pd.api.types.is_numeric_dtype(df_q[col]):
        issues.append(f"Questionnaire column '{col}' expected numeric, got {df_q[col].dtype}")

EXPECTED_NUMERIC_EYE = ["Saccade count", "Fixation count", "Blink count",
                         "Reading time", "Comprehension% (questions correct)"]
for col in EXPECTED_NUMERIC_EYE:
    if col in df_eye.columns and not pd.api.types.is_numeric_dtype(df_eye[col]):
        issues.append(f"Eye metrics column '{col}' expected numeric, got {df_eye[col].dtype}")

print("Dtypes OK for checked columns" if not any("expected numeric" in i for i in issues) else "Dtype issues found — see summary")


# ── 7. VALUE RANGE SANITY ────────────────────────────────────
print("\n=== VALUE RANGES ===")
# VAS scores (questionnaire text questionnaire fields) should be 0–100
vas_cols = [c for c in df_q.columns if "Text questionnaire" in c]
for col in vas_cols:
    if df_q[col].between(0, 100).all() == False:
        issues.append(f"VAS column '{col}' has values outside [0, 100]")

# Comprehension should be 0–100%
if not df_eye["Comprehension% (questions correct)"].dropna().between(0, 100).all():
    issues.append("Comprehension% has values outside [0, 100]")

# Reading time and saccade counts should be positive
for col in ["Reading time", "Saccade count", "Fixation count"]:
    if (df_eye[col] < 0).any():
        issues.append(f"Eye metrics '{col}' has negative values")

print("Value ranges checked")


# ── 8. SUMMARY ───────────────────────────────────────────────
print("\n" + "="*50)
if issues:
    print(f"⚠  {len(issues)} ISSUE(S) FOUND:")
    for i, msg in enumerate(issues, 1):
        print(f"  [{i}] {msg}")
else:
    print("✓  All checks passed — data looks consistent")
print("="*50)

=== SHAPE ===
Questionnaire : 12 rows × 65 cols
Eye metrics   : 66 rows × 19 cols

=== PARTICIPANT ID ALIGNMENT ===
Shared participants : 12
Only in questionnaire : none
Only in eye metrics   : ['P013', 'P014', 'P015', 'P016', 'P017', 'P018', 'P019', 'P020', 'P021', 'P022']

=== CONDITION COMPLETENESS ===
Condition counts per participant:
Condition       eyetracked  full  none
Participant ID                        
P001                     1     1     1
P002                     1     1     1
P003                     1     1     1
P004                     1     1     1
P005                     1     1     1
P006                     1     1     1
P007                     1     1     1
P008                     1     1     1
P009                     1     1     1
P010                     1     1     1
P011                     1     1     1
P012                     1     1     1
P013                     1     1     1
P014                     1     1     1
P015                     1     1   

***Check data for skews and which measurements might need a log1p transform to survive normality assumption if too positive skew***

***Reading section***
* Reading time RM-anova
* Reading comprehension accuracy RM-anova with Greenhouse-Geisser if Mauchly denies sphericity
* Saccades count RM-anova
* Saccades duration RM-anova
* Fixations count RM-anova
* Fixations duration RM-anova
* Blinks count RM-anova

*Reading time RM-anova*

In [46]:
df_rt = df_eye[["Participant ID", "Condition", "Reading time"]].dropna()
CONDITIONS = sorted(df_rt["Condition"].unique())

result = af.run_rm_anova_or_friedman(
    df            = df_rt,
    subject_col   = "Participant ID",
    condition_col = "Condition",
    value_col     = "Reading time",
    alpha         = 0.05
)

ac = result["assumptions_checked"]
sph = ac.get("sphericity_details", {})

# ── [ANALYSIS] block ─────────────────────────────────────────
print(f"[ANALYSIS] Reading Time by Condition")
print(f"[ANALYSIS] Subjects: {result['n_subjects_final']} used, "
      f"{result['n_subjects_initial'] - result['n_subjects_final']} dropped "
      f"(listwise deletion of incomplete cases)")
print(f"[ANALYSIS] Conditions: {CONDITIONS}")
if result["dropped_subjects"]:
    print(f"[ANALYSIS] Dropped subjects: {result['dropped_subjects']}")

# ── [ASSUMPTION] block ───────────────────────────────────────
print()
print(f"[ASSUMPTION] Normality (Shapiro-Wilk on pairwise differences):")
normality_notes = [n for n in ac["notes"] if "Normality" in n or "normality" in n or "Shapiro" in n]
if normality_notes:
    for n in normality_notes:
        print(f"[ASSUMPTION]   {n}")
else:
    print(f"[ASSUMPTION]   All pairwise differences passed normality (p >= 0.05)")

print(f"[ASSUMPTION] Sphericity (Mauchly's test):")
print(f"[ASSUMPTION]   W = {sph.get('mauchly_stat', float('nan')):.6f}, "
      f"p = {sph.get('mauchly_p', float('nan')):.6f}")
print(f"[ASSUMPTION]   Sphericity assumed: {sph.get('sphericity_assumed', 'unknown')}")

if ac["normality_violated"] or not ac["sphericity_verified"]:
    print(f"[ASSUMPTION] One or more assumptions violated — falling back to Friedman test")
else:
    print(f"[ASSUMPTION] All assumptions met — proceeding with RM ANOVA")

# ── [RESULT] block ───────────────────────────────────────────
print()
print(f"[RESULT] Test used: {result['test']}")

if result["test"] == "RM ANOVA":
    df_b, df_e = result["df"]
    print(f"[RESULT] F({df_b}, {df_e}) = {result['statistic']:.4f}, p = {result['p_value']:.4f}")
else:
    print(f"[RESULT] chi2({result['df']}) = {result['statistic']:.4f}, p = {result['p_value']:.4f}")

if result["significant"]:
    print(f"[RESULT] Significant effect of condition on reading time (p < 0.05)")
else:
    print(f"[RESULT] No significant effect of condition on reading time (p >= 0.05)")

# ── [RESULT] post-hoc ────────────────────────────────────────
if result["post_hoc"] is not None:
    ph = result["post_hoc"]
    label_map = {
        "g1 vs g2": f"{CONDITIONS[0]} vs {CONDITIONS[1]}",
        "g1 vs g3": f"{CONDITIONS[0]} vs {CONDITIONS[2]}",
        "g2 vs g3": f"{CONDITIONS[1]} vs {CONDITIONS[2]}",
    }
    print()
    print(f"[RESULT] Post-hoc: {ph['method']}, Bonferroni alpha = {ph['bonferroni_alpha']:.4f}")
    for pair, vals in ph["comparisons"].items():
        label    = label_map.get(pair, pair)
        stat_key = "t_statistic" if "t_statistic" in vals else "z_statistic"
        stat_name = "t" if "t_statistic" in vals else "W"
        sig      = "significant" if vals["significant"] else "not significant"
        print(f"[RESULT]   {label}: {stat_name} = {vals[stat_key]:.4f}, "
              f"p = {vals['p_value']:.4f} ({sig})")
else:
    print(f"[RESULT] No post-hoc tests run (main test not significant)")


MAUCHLY'S TEST FOR SPHERICITY
Mauchly's W statistic: 0.897943
P-value: 0.340790
✓ NOT SIGNIFICANT (p >= 0.05): Sphericity is ASSUMED
  → Interpretation: The assumption of equal variances of differences
                   between condition pairs is satisfied.
  → Recommendation: RM ANOVA can be used safely.
[ANALYSIS] Reading Time by Condition
[ANALYSIS] Subjects: 22 used, 0 dropped (listwise deletion of incomplete cases)
[ANALYSIS] Conditions: ['eyetracked', 'full', 'none']

[ASSUMPTION] Normality (Shapiro-Wilk on pairwise differences):
[ASSUMPTION]   Normality violated for g1-g2 (Shapiro-Wilk p=0.0005)
[ASSUMPTION] Sphericity (Mauchly's test):
[ASSUMPTION]   W = 0.897943, p = 0.340790
[ASSUMPTION]   Sphericity assumed: True
[ASSUMPTION] One or more assumptions violated — falling back to Friedman test

[RESULT] Test used: Friedman
[RESULT] chi2(2) = 27.2727, p = 0.0000
[RESULT] Significant effect of condition on reading time (p < 0.05)

[RESULT] Post-hoc: Wilcoxon signed-rank test wit

*Reading comprehension question results RM anova*

In [47]:
#FIll out later

*Saccades count RM ANOVA*

*Saccades duration RM ANOVA*

*Fixation count RM ANOVA*

*Fixation duration RM ANOVA*

*Blinks count RM anova*

**Detectability section**
* Identification accuracy Chi-square 3x3 confusion matrix
* Identification response time (ms) RM-anova

**Questionnaire descriptive and explorative statistics**
Attempting to find patterns across specifics in the questionnaire data, such as trends in eye strain types (whether each condition causes dryness, doubled vision, or similar profiles to each other but at different scales)